# DDL: `dbspend360_total_all_purpose_spends`

Creates the final per-cluster / per-user / per-day spend rollup for all-purpose clusters
(`cluster_source IN ('UI','API')`), combining cloud and Databricks costs.

Sibling of `dbspend360_total_job_spends`, but keyed on `(cluster_id, user_id, usage_date)`
instead of `(cluster_id, job_id, run_id, usage_date)`. `data_security_mode` is denormalized
from `system.compute.clusters` so the UI can render attribution-quality badges
(Dedicated / Shared / Legacy / Unknown) without an extra join.

**Widgets**
- `catalog` - target Unity Catalog name
- `schema`  - target schema name within `catalog`

In [ ]:
dbutils.widgets.text("catalog", "", "Catalog")
dbutils.widgets.text("schema", "", "Schema")

In [ ]:
catalog = dbutils.widgets.get("catalog").strip()
schema = dbutils.widgets.get("schema").strip()

if not catalog or not schema:
    raise ValueError("Both `catalog` and `schema` widgets must be set.")

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{schema}")

In [ ]:
%sql
CREATE TABLE IF NOT EXISTS ${catalog}.${schema}.dbspend360_total_all_purpose_spends (
  cluster_id          STRING,
  user_id             STRING,
  usage_date          DATE,
  cloud_cost          DOUBLE,
  compute_cost        DOUBLE,
  storage_cost        DOUBLE,
  network_cost        DOUBLE,
  other_cost          DOUBLE,
  databricks_cost     DOUBLE,
  currency            STRING,
  total_cost          DOUBLE,
  data_security_mode  STRING,
  workspace_covered   BOOLEAN,
  created_at          TIMESTAMP,
  updated_at          TIMESTAMP
)
CLUSTER BY AUTO

In [ ]:
dbutils.notebook.exit(f"{catalog}.{schema}.dbspend360_total_all_purpose_spends")